In [0]:
%run ./01_logging_utils.py

In [0]:

from pyspark.sql import functions as F


def run_validation_checks():
    print("=" * 55)
    print("CRYPTOCURRENCY PIPELINE VALIDATION")
    print("=" * 55)

    bronze_df = spark.table(T_BRONZE)
    silver_df = spark.table(T_SILVER)
    quarantine_df = spark.table(T_QUAR)

    # 1. Row counts
    bronze_count = bronze_df.count()
    silver_count = silver_df.count()
    quarantine_count = quarantine_df.count()

    print("\n1. ROW COUNTS")
    print(f"Bronze rows: {bronze_count}")
    print(f"Silver rows: {silver_count}")
    print(f"Quarantined rows: {quarantine_count}")

    # 2. Distinct coins
    print("\n2. DISTINCT COINS")
    print(
        "Bronze coins:",
        bronze_df.select("coin_id").distinct().count()
    )
    print(
        "Silver coins:",
        silver_df.select("coin_id").distinct().count()
    )

    # 3. Bronze duplicate keys
    bronze_duplicates = (
        bronze_df
        .groupBy("coin_id", "ts_ms")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print("\n3. BRONZE DUPLICATES")
    print(f"Duplicate coin/timestamp keys: {bronze_duplicates}")

    # 4. Silver duplicate keys
    silver_duplicates = (
        silver_df
        .groupBy("coin_id", "event_ts")
        .count()
        .filter(F.col("count") > 1)
        .count()
    )

    print("\n4. SILVER DUPLICATES")
    print(f"Duplicate coin/timestamp keys: {silver_duplicates}")

    # 5. Invalid Silver values
    invalid_silver = silver_df.filter(
        F.col("coin_id").isNull()
        | F.col("event_ts").isNull()
        | F.col("price").isNull()
        | (F.col("price") < 0)
        | (
            F.col("market_cap").isNotNull()
            & (F.col("market_cap") < 0)
        )
        | (
            F.col("total_volume").isNotNull()
            & (F.col("total_volume") < 0)
        )
    ).count()

    print("\n5. INVALID SILVER RECORDS")
    print(f"Invalid records: {invalid_silver}")

    # 6. Null checks in Bronze
    bronze_nulls = bronze_df.filter(
        F.col("coin_id").isNull()
        | F.col("ts_ms").isNull()
        | F.col("load_timestamp").isNull()
    ).count()

    print("\n6. BRONZE REQUIRED-FIELD NULLS")
    print(f"Records with null required fields: {bronze_nulls}")

    # 7. Timestamp range
    print("\n7. SILVER TIMESTAMP RANGE")
    silver_df.select(
        F.min("event_ts").alias("earliest_timestamp"),
        F.max("event_ts").alias("latest_timestamp")
    ).show(truncate=False)

    # 8. Recent pipeline runs
    print("\n8. RECENT PIPELINE RUNS")
    spark.table(T_LOGS).orderBy(
        F.col("start_time").desc()
    ).show(10, truncate=False)

    # 9. Validation summary
    checks = [
        ("Bronze has records", bronze_count > 0),
        ("Silver has records", silver_count > 0),
        ("No Bronze duplicate keys", bronze_duplicates == 0),
        ("No Silver duplicate keys", silver_duplicates == 0),
        ("No invalid Silver records", invalid_silver == 0),
        ("No null Bronze required fields", bronze_nulls == 0)
    ]

    print("\n" + "=" * 55)
    print("VALIDATION SUMMARY")
    print("=" * 55)

    for check_name, passed in checks:
        status = "PASS" if passed else "FAIL"
        print(f"{status}: {check_name}")

    failed_checks = sum(
        1 for _, passed in checks if not passed
    )

    print(f"\nPassed: {len(checks) - failed_checks}/{len(checks)}")
    print(f"Failed: {failed_checks}/{len(checks)}")

    return checks


In [0]:
validation_results = run_validation_checks()

In [0]:
validation_df = spark.createDataFrame(
    validation_results,
    ["check_name", "passed"]
)

display(validation_df)

In [0]:
display(spark.sql("""
    SELECT 'Bronze' AS layer, COUNT(*) AS total_rows
    FROM crypto.bronze.market_data_raw

    UNION ALL

    SELECT 'Silver', COUNT(*)
    FROM crypto.silver.market_data_clean

    UNION ALL

    SELECT 'Quarantine', COUNT(*)
    FROM crypto.bronze.quarantine
"""))

In [0]:
display(spark.sql("""
    SELECT layer, status, COUNT(*) AS run_count
    FROM crypto.ops.pipeline_execution_logs
    GROUP BY layer, status
    ORDER BY layer, status
"""))

In [0]:
display(spark.sql("""
    SELECT
        run_id,
        layer,
        file_or_params,
        start_time,
        status,
        error_message
    FROM crypto.ops.pipeline_execution_logs
    WHERE status = 'FAILED'
    ORDER BY start_time DESC
"""))

the above failure was issue before but we resolved it later so now its showing just as history

In [0]:
display(spark.sql("""
    SELECT
        'Bronze' AS layer,
        COUNT(*) AS total_rows,
        COUNT(DISTINCT coin_id) AS unique_coins,
        COUNT(*) - COUNT(DISTINCT
            CONCAT(coin_id, '_', CAST(ts_ms AS STRING))
        ) AS duplicate_key_count
    FROM crypto.bronze.market_data_raw

    UNION ALL

    SELECT
        'Silver',
        COUNT(*),
        COUNT(DISTINCT coin_id),
        COUNT(*) - COUNT(DISTINCT
            CONCAT(coin_id, '_', CAST(event_ts AS STRING))
        )
    FROM crypto.silver.market_data_clean
"""))

In [0]:
%sql
SELECT
    run_id,
    layer,
    file_or_params,
    start_time,
    end_time,
    status,
    rows_inserted,
    rows_updated,
    error_message
FROM crypto.ops.pipeline_execution_logs
ORDER BY start_time DESC;

In [0]:
%sql

SELECT
    run_id,
    layer,
    file_or_params,
    rows_inserted,
    rows_updated,
    status
FROM crypto.ops.pipeline_execution_logs
WHERE layer IN ('backfill', 'schema_drift')
ORDER BY start_time DESC;


In [0]:
%sql

SELECT
    COUNT(*) AS rows_in_date_range,
    COUNT(DISTINCT coin_id) AS distinct_coins
FROM crypto.silver.market_data_clean
WHERE event_date >= DATE '2026-09-04'
  AND event_date <= DATE '2026-10-04';
